# Module C: JEPA-Augmented BC + REPA (C1–C5)

**Behavior Cloning with JEPA Data Augmentation & REPA-Style Intermediate Alignment**

| # | Task | Detail |
|---|------|--------|
| C1 | BC policy baseline | Train MLP head (DINOv2 embedding → 7D action) on real data |
| C2 | Synthetic trajectory generation | Perturb embeddings, roll out predictor with expert actions, store $(z', a)$ pairs |
| C3 | JEPA-augmented BC training | Train BC on real + synthetic at $α \in \{0.0,0.25,0.5,0.75,1.0\}$, ablate $σ \in \{0.01,0.05,0.1\}$ |
| C4 | REPA alignment | Cache DINOv2 intermediate layer features (all 12 blocks), add projection heads, train with $λ \in \{0.1,0.3,0.5\}$ |
| C5 | REPA ablation results | Cosine sim, rollout drift, CEM success with/without REPA |

**Prerequisites**: Module A predictor checkpoints at `outputs/module_a_transformer_K1_T8.pt`. Cached embeddings in `data/embeddings/`.


## Section 1: Setup & Imports


In [ ]:
import sys, subprocess

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['torch>=2.0', 'matplotlib', 'scikit-learn', 'tqdm', 'numpy', 'pandas', 'scipy', 'pyarrow', 'timm']:
    pip_install(pkg)

import os, math, random, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# -- Paths (Colab-aware) --
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/jepa_action")
    print("Mounted Google Drive. Make sure data/ is at MyDrive/jepa_action/data/")
except ImportError:
    ROOT = Path(os.getcwd()).resolve()
    if not (ROOT / "data" / "embeddings").exists():
        if (ROOT.parent / "data" / "embeddings").exists():
            ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
EMBED_DIR = DATA_DIR / "embeddings"
PARQUET_DIR = DATA_DIR / "robomimic-ph-lift-image" / "data" / "chunk-000"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Config
class Config:
    # Core (from module_a/b)
    embed_dim = 384
    action_dim = 7
    d_model = 256
    n_heads = 4
    n_layers_tr = 4
    batch_size = 64
    lr = 1e-4
    weight_decay = 0.05
    warmup_steps = 200
    max_epochs = 50
    use_amp = True
    dropout_p = 0.2
    seed = 42
    K_values = [1, 5, 10, 20]
    best_T = 8
    n_train_episodes = 160
    # BC
    bc_hidden = [256, 128]
    bc_epochs = 30
    bc_batch_size = 64
    # Synthetic generation
    N_perturb = 10
    sigma_values = [0.01, 0.05, 0.1]
    rollout_K = 1
    rollout_T = 8
    # JEPA-augmented BC
    alpha_values = [0.0, 0.25, 0.5, 0.75, 1.0]
    # REPA
    dino_all_layers = 12
    repa_layer_map = [2, 5, 8, 11]
    repa_lambda_values = [0.1, 0.3, 0.5]
    repa_epochs = 50

cfg = Config()

# Device & reproducibility
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cudnn.benchmark = True

print('Setup complete.')


## Section 2: Load Cached Data

Load pre-computed embeddings, actions, and per-episode data from Parquet files. Build episode-to-embedding-index mapping.


In [ ]:
# Load cached embeddings and actions
train_emb = torch.load(EMBED_DIR / 'embeddings_train.pt').float()
val_emb = torch.load(EMBED_DIR / 'embeddings_val.pt').float()
train_act = torch.load(EMBED_DIR / 'actions_train.pt').float()
val_act = torch.load(EMBED_DIR / 'actions_val.pt').float()
norm_stats = torch.load(EMBED_DIR / 'norm_stats.pt')

print(f'Train embeddings: {train_emb.shape}')
print(f'Val embeddings:   {val_emb.shape}')
print(f'Train actions:    {train_act.shape}')

# Load all Parquet files for episode boundaries
parquet_files = sorted(PARQUET_DIR.glob('episode_*.parquet'))
print(f'Found {len(parquet_files)} parquet episodes')

all_ep_actions = []
all_ep_n_frames = []

for pf in tqdm(parquet_files, desc='Loading parquets'):
    df = pd.read_parquet(pf)
    actions_raw = np.stack(df['action'].values).astype(np.float32)
    all_ep_actions.append(torch.tensor(actions_raw[::2]))
    all_ep_n_frames.append(len(actions_raw[::2]))

# Verify train/val split
train_frames_total = sum(all_ep_n_frames[:cfg.n_train_episodes])
val_frames_total = sum(all_ep_n_frames[cfg.n_train_episodes:])
assert train_frames_total == train_emb.shape[0], f'Train mismatch: {train_frames_total} vs {train_emb.shape[0]}'
assert val_frames_total == val_emb.shape[0], f'Val mismatch: {val_frames_total} vs {val_emb.shape[0]}'
print(f'Train: {train_frames_total} frames in {cfg.n_train_episodes} episodes')
print(f'Val:   {val_frames_total} frames in {len(parquet_files) - cfg.n_train_episodes} episodes')

# Build episode-to-embedding-index mapping
train_offset = 0
val_offset = 0
ep_to_emb_start = {}

for ep_idx in range(len(parquet_files)):
    n_frames = all_ep_n_frames[ep_idx]
    if ep_idx < cfg.n_train_episodes:
        ep_to_emb_start[ep_idx] = ('train', train_offset)
        train_offset += n_frames
    else:
        ep_to_emb_start[ep_idx] = ('val', val_offset)
        val_offset += n_frames

def get_emb_idx(ep_idx, local_frame_idx):
    split_name, start = ep_to_emb_start[ep_idx]
    return split_name, start + local_frame_idx

print('Data loaded and verified.')


## Section 3: C1 — BC Policy Baseline

Train a simple MLP policy that maps a DINOv2 embedding (384D) to a 7D action via MSE loss.
This is pure Behavior Cloning on real demonstration data.

Architecture: $384 \to 256 \to 128 \to 7$ with ReLU activations.


In [ ]:
class BCDataset(Dataset):
    """Dataset for Behavior Cloning: pairs embeddings with corresponding actions."""
    def __init__(self, embeddings, actions, ep_n_frames):
        """
        embeddings: (N_total, 384) tensor
        actions:    (N_total, 7) tensor
        ep_n_frames: list of frame counts per episode
        """
        self.embeddings = embeddings
        self.actions = actions
        self.samples = []
        offset = 0
        for n_frames in ep_n_frames:
            for i in range(n_frames):
                self.samples.append((offset + i, offset + i))
            offset += n_frames

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        emb_idx, act_idx = self.samples[idx]
        return self.embeddings[emb_idx], self.actions[act_idx]


# Build BC datasets
train_ep_frames = all_ep_n_frames[:cfg.n_train_episodes]
val_ep_frames = all_ep_n_frames[cfg.n_train_episodes:]
bc_train_ds = BCDataset(train_emb, train_act, train_ep_frames)
bc_val_ds = BCDataset(val_emb, val_act, val_ep_frames)

bc_train_ldr = DataLoader(bc_train_ds, batch_size=cfg.bc_batch_size, shuffle=True,
                         num_workers=0, pin_memory=(device.type=='cuda'), drop_last=True)
bc_val_ldr = DataLoader(bc_val_ds, batch_size=cfg.bc_batch_size, shuffle=False,
                        num_workers=0, pin_memory=(device.type=='cuda'))

print(f'BC train samples: {len(bc_train_ds)}, val samples: {len(bc_val_ds)}')


In [ ]:
class BCPolicy(nn.Module):
    """Simple MLP policy: embedding (384) → action (7)."""
    def __init__(self, embed_dim=384, action_dim=7, hidden_dims=[256, 128]):
        super().__init__()
        layers = []
        in_dim = embed_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(in_dim, h), nn.ReLU()])
            in_dim = h
        layers.append(nn.Linear(in_dim, action_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)

bc_model = BCPolicy(embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
                     hidden_dims=cfg.bc_hidden).to(device)
print(f'BC policy params: {sum(p.numel() for p in bc_model.parameters()):,}')


In [ ]:
def bc_train_one_epoch(model, loader, optimizer, scaler, global_step, total_steps):
    model.train()
    epoch_loss = 0.0
    for z, a_target in loader:
        z = z.to(device)
        a_target = a_target.to(device)
        lr = get_lr(global_step, cfg.warmup_steps, total_steps, cfg.lr)
        for pg in optimizer.param_groups:
            pg['lr'] = lr
        if cfg.use_amp and device.type == 'cuda':
            with autocast():
                a_pred = model(z)
                loss = F.mse_loss(a_pred, a_target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            a_pred = model(z)
            loss = F.mse_loss(a_pred, a_target)
            loss.backward()
            optimizer.step()
        optimizer.zero_grad()
        epoch_loss += loss.item()
        global_step += 1
    return epoch_loss / len(loader), global_step


@torch.no_grad()
def bc_compute_val_mse(model, loader):
    model.eval()
    mse_list = []
    for z, a_target in loader:
        z = z.to(device)
        a_target = a_target.to(device)
        a_pred = model(z)
        mse = F.mse_loss(a_pred, a_target).item()
        mse_list.append(mse)
    return np.mean(mse_list)


def bc_train_model(model, train_ldr, val_ldr, epochs=None, verbose=True):
    epochs = epochs or cfg.bc_epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = GradScaler(enabled=(cfg.use_amp and device.type=='cuda'))
    total_steps = epochs * len(train_ldr)
    gs = 0
    train_losses, val_mses = [], []
    best_mse = float('inf')
    best_state = None
    for epoch in range(1, epochs + 1):
        avg_loss, gs = bc_train_one_epoch(model, train_ldr, optimizer, scaler, gs, total_steps)
        val_mse = bc_compute_val_mse(model, val_ldr)
        train_losses.append(avg_loss)
        val_mses.append(val_mse)
        if val_mse < best_mse:
            best_mse = val_mse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if verbose and (epoch % max(1, epochs // 10) == 0 or epoch == 1 or epoch == epochs):
            print(f'E {epoch:3d}/{epochs} | loss={avg_loss:.4f} | val_mse={val_mse:.4f} | best={best_mse:.4f}')
    model.load_state_dict(best_state)
    return {'train_losses': train_losses, 'val_mses': val_mses, 'best_mse': best_mse}


# Re-use LR scheduler from module_a
def cosine_loss(pred, target):
    return 1.0 - F.cosine_similarity(pred, target, dim=-1).mean()

def get_lr(step, warmup_steps, total_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

print('BC training utilities ready.')


In [ ]:
print('Training BC baseline on real data...')
print('-' * 55)

bc_baseline = BCPolicy(embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
                        hidden_dims=cfg.bc_hidden).to(device)
bc_result = bc_train_model(bc_baseline, bc_train_ldr, bc_val_ldr, epochs=cfg.bc_epochs)

print(f'\nBC baseline best val MSE: {bc_result["best_mse"]:.6f}')

# Save checkpoint
bc_ckpt = {
    'model_state_dict': bc_baseline.state_dict(),
    'val_mse': bc_result['best_mse'],
    'config': {k: v for k, v in vars(cfg).items() if not k.startswith('__')}
}
torch.save(bc_ckpt, OUTPUT_DIR / 'module_c_bc_baseline.pt')
print('Saved outputs/module_c_bc_baseline.pt')

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(bc_result['train_losses'], label='Train MSE')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE')
ax1.set_title('C1: BC Training Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(bc_result['val_mses'], color='orange', label='Val MSE')
ax2.axhline(y=bc_result['best_mse'], color='r', linestyle='--', label=f'Best={bc_result["best_mse"]:.4f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('MSE')
ax2.set_title('C1: BC Validation MSE'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'c1_bc_baseline.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 4: C2 — Synthetic Trajectory Generation

For each training episode, perturb the initial embedding $z_0$ with Gaussian noise $\epsilon \sim \mathcal{N}(0, \sigma^2 I)$, then autoregressively roll out the JEPA predictor with expert actions to generate synthetic $(\hat{z}_t, a_t)$ pairs.

Ablate $\sigma \in \{0.01, 0.05, 0.1\}$ with $N_\text{perturb}=10$ per episode.


In [ ]:
# Re-define Transformer predictor (same as Module A/B)
class ActionMLP(nn.Module):
    def __init__(self, action_dim=7, hidden_dim=128, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(action_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, a):
        return self.net(a)


class TransformerPredictor(nn.Module):
    def __init__(self, embed_dim=384, action_dim=7, d_model=256,
                 n_layers=4, n_heads=4, max_seq_len=64, dropout=0.2):
        super().__init__()
        self.d_model = d_model
        self.embed_proj = nn.Linear(embed_dim, d_model)
        self.action_mlp = ActionMLP(action_dim, hidden_dim=128, out_dim=d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, max_seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, batch_first=True,
            dropout=dropout, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.predictor = nn.Linear(d_model, embed_dim)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, embeds, actions):
        if embeds.dim() == 2:
            embeds = embeds.unsqueeze(1); actions = actions.unsqueeze(1)
        B, T, _ = embeds.shape
        x = self.embed_proj(embeds) + self.action_mlp(actions)
        x = x + self.pos_embed[:, :T, :]
        causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        x = self.transformer(x, mask=causal_mask)
        return self.predictor(x[:, -1, :])


# Load best K=1, T=8 predictor
ckpt_path = OUTPUT_DIR / f'module_a_transformer_K{cfg.rollout_K}_T{cfg.rollout_T}.pt'
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
rollout_predictor = TransformerPredictor(
    embed_dim=cfg.embed_dim, action_dim=cfg.action_dim, d_model=cfg.d_model,
    n_layers=cfg.n_layers_tr, n_heads=cfg.n_heads, dropout=cfg.dropout_p,
).to(device)
rollout_predictor.load_state_dict(ckpt['model_state_dict'])
rollout_predictor.eval()
print(f'Loaded predictor K={cfg.rollout_K} T={cfg.rollout_T} (val_cos={ckpt.get("val_cos", "?")})')


In [ ]:
@torch.no_grad()
def generate_synthetic_episode(predictor, z_0, expert_actions, T, sigma, N_perturb, act_noise=0.0):
    """
    Generate N_perturb synthetic trajectories from a perturbed initial state.
    
    Args:
        predictor: JEPA predictor model (K=1 recommended)
        z_0: initial embedding (384,)
        expert_actions: (ep_len, 7) tensor of expert actions
        T: context window
        sigma: std of Gaussian noise for perturbation
        N_perturb: number of perturbations
    Returns:
        synthetic_z: (N_perturb, ep_len, 384) synthetic embeddings
        synthetic_a: (ep_len, 7) expert actions (repeated)
    """
    ep_len = len(expert_actions)
    synthetic_z = torch.zeros(N_perturb, ep_len, cfg.embed_dim)
    
    for n in range(N_perturb):
        # Perturb initial embedding
        z_cur = z_0 + torch.randn(cfg.embed_dim) * sigma
        z_cur = z_cur.to(device)
        synthetic_z[n, 0] = z_cur.cpu()
        
        # Build initial context: repeat z_0 T times
        z_ctx = z_cur.unsqueeze(0).unsqueeze(0).repeat(1, T, 1)
        a_ctx = expert_actions[:1].unsqueeze(0).repeat(1, T, 1).to(device)
        
        for t in range(1, ep_len):
            # Predict next z
            pred = predictor(z_ctx, a_ctx)  # (1, 384)
            # Slide context
            z_ctx = torch.cat([z_ctx[:, 1:, :], pred.unsqueeze(1)], dim=1)
            a_new = expert_actions[t:t+1].unsqueeze(0).repeat(1, T, 1).to(device)
            a_ctx = a_new
            synthetic_z[n, t] = pred.squeeze(0).cpu()
    
    return synthetic_z, expert_actions


print('Synthetic generation function ready.')


In [ ]:
# Generate synthetic trajectories for all sigma values
# Save each sigma's results separately

for sigma in cfg.sigma_values:
    print(f'\nGenerating synthetic trajectories with sigma={sigma}...')
    all_synthetic_z = []
    all_synthetic_a = []
    
    for ep_idx in tqdm(range(cfg.n_train_episodes), desc=f'sigma={sigma}'):
        split_name, start = ep_to_emb_start[ep_idx]
        emb = train_emb if split_name == 'train' else val_emb
        n_frames = all_ep_n_frames[ep_idx]
        if n_frames < 2:
            continue
        z_0 = emb[start]
        expert_actions = all_ep_actions[ep_idx]  # raw actions (not normalized)
        
        syn_z, syn_a = generate_synthetic_episode(
            rollout_predictor, z_0, expert_actions,
            T=cfg.rollout_T, sigma=sigma, N_perturb=cfg.N_perturb
        )
        all_synthetic_z.append(syn_z)
        all_synthetic_a.append(syn_a)
    
    # Save
    out_path = EMBED_DIR / f'synthetic_sigma{sigma}.pt'
    torch.save({'z': all_synthetic_z, 'a': all_synthetic_a}, out_path)
    print(f'  Saved {out_path} ({len(all_synthetic_z)} episodes x {cfg.N_perturb} perturb)')

print('\nSynthetic trajectory generation complete.')


In [ ]:
# Quality check: mean cosine similarity between synthetic and real embeddings over rollout steps
fig, ax = plt.subplots(figsize=(8, 5))

for sigma in cfg.sigma_values:
    data = torch.load(EMBED_DIR / f'synthetic_sigma{sigma}.pt', weights_only=False)
    syn_z_list = data['z']  # list of (N_perturb, ep_len, 384)
    
    all_cos = []
    max_step = 30
    for ep_idx, syn_z in enumerate(syn_z_list[:10]):
        split_name, start = ep_to_emb_start[ep_idx]
        emb = train_emb if split_name == 'train' else val_emb
        real_z = emb[start:start+syn_z.shape[1]]
        steps = min(syn_z.shape[1], real_z.shape[0], max_step)
        cos_per_step = []
        for t in range(steps):
            cs = F.cosine_similarity(syn_z[:, t], real_z[t:t+1].expand(syn_z.shape[0], -1), dim=-1).mean()
            cos_per_step.append(cs.item())
        all_cos.append(cos_per_step)
    
    # Average across episodes
    min_len = min(len(c) for c in all_cos)
    avg_cos = np.mean([c[:min_len] for c in all_cos], axis=0)
    ax.plot(range(min_len), avg_cos, marker='.', label=f'σ={sigma}')

ax.set_xlabel('Rollout step')
ax.set_ylabel('Mean cos_sim(synthetic, real)')
ax.set_title('C2: Synthetic Trajectory Quality (cos_sim vs step)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'c2_synthetic_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print('Quality check done.')


## Section 5: C3 — JEPA-Augmented BC Training

Mix real and synthetic data at ratios $\alpha \in \{0.0, 0.25, 0.5, 0.75, 1.0\}$ and train BC policies.
First select the best $\sigma$ (lowest val MSE at $\alpha=0.5$), then sweep $\alpha$ with that $\sigma$.


In [ ]:
class MixedBCDataset(Dataset):
    """Mix real and synthetic BC samples at ratio alpha.
    alpha=0.0: 100% real; alpha=1.0: 100% synthetic.
    Batch composition: each __getitem__ returns either real or synthetic based on alpha."""
    def __init__(self, real_embeddings, real_actions, ep_n_frames,
                 synthetic_z_list, synthetic_a_list, alpha=0.5):
        self.alpha = alpha
        # Index real samples
        self.real_samples = []
        offset = 0
        for n_frames in ep_n_frames:
            for i in range(n_frames):
                self.real_samples.append((offset + i, offset + i))
            offset += n_frames
        self.real_emb = real_embeddings
        self.real_act = real_actions
        # Flatten synthetic samples
        self.syn_samples = []
        for syn_z, syn_a in zip(synthetic_z_list, synthetic_a_list):
            N_perturb, ep_len, _ = syn_z.shape
            for n in range(N_perturb):
                for t in range(ep_len):
                    self.syn_samples.append((syn_z[n, t], syn_a[t]))
        print(f'Real samples: {len(self.real_samples)}, Synthetic: {len(self.syn_samples)}, α={alpha}')

    def __len__(self):
        return max(len(self.real_samples), len(self.syn_samples))

    def __getitem__(self, idx):
        if torch.rand(1).item() < self.alpha and len(self.syn_samples) > 0:
            si = idx % len(self.syn_samples)
            return self.syn_samples[si]
        else:
            ri = idx % len(self.real_samples)
            emb_i, act_i = self.real_samples[ri]
            return self.real_emb[emb_i], self.real_act[act_i]

print('MixedBCDataset ready.')


In [ ]:
# Quick sigma selection: train BC at alpha=0.5 for each sigma, pick best
sigma_mse = {}

for sigma in cfg.sigma_values:
    print(f'\nTesting σ={sigma} at α=0.5...')
    data = torch.load(EMBED_DIR / f'synthetic_sigma{sigma}.pt', weights_only=False)
    syn_z_list, syn_a_list = data['z'], data['a']
    
    mixed_ds = MixedBCDataset(train_emb, train_act, train_ep_frames,
                              syn_z_list, syn_a_list, alpha=0.5)
    mixed_ldr = DataLoader(mixed_ds, batch_size=cfg.bc_batch_size, shuffle=True,
                           num_workers=0, pin_memory=(device.type=='cuda'), drop_last=True)
    
    model = BCPolicy(embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
                     hidden_dims=cfg.bc_hidden).to(device)
    result = bc_train_model(model, mixed_ldr, bc_val_ldr, epochs=15, verbose=False)
    sigma_mse[sigma] = result['best_mse']
    print(f'  σ={sigma}: best val MSE = {result["best_mse"]:.6f}')

best_sigma = min(sigma_mse, key=sigma_mse.get)
print(f'\nBest sigma: {best_sigma} (MSE={sigma_mse[best_sigma]:.6f})')


In [ ]:
# Full alpha sweep with best sigma
data = torch.load(EMBED_DIR / f'synthetic_sigma{best_sigma}.pt', weights_only=False)
syn_z_list, syn_a_list = data['z'], data['a']

aug_results = {}

for alpha in cfg.alpha_values:
    print(f'\nTraining BC with α={alpha}...')
    print('-' * 40)
    
    mixed_ds = MixedBCDataset(train_emb, train_act, train_ep_frames,
                              syn_z_list, syn_a_list, alpha=alpha)
    mixed_ldr = DataLoader(mixed_ds, batch_size=cfg.bc_batch_size, shuffle=True,
                           num_workers=0, pin_memory=(device.type=='cuda'), drop_last=True)
    
    model = BCPolicy(embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
                     hidden_dims=cfg.bc_hidden).to(device)
    result = bc_train_model(model, mixed_ldr, bc_val_ldr, epochs=cfg.bc_epochs)
    aug_results[alpha] = result
    
    # Save checkpoint
    ckpt = {
        'model_state_dict': model.state_dict(),
        'val_mse': result['best_mse'],
        'alpha': alpha, 'sigma': best_sigma,
    }
    torch.save(ckpt, OUTPUT_DIR / f'module_c_bc_aug_alpha{alpha}.pt')

print('\nAlpha sweep complete.')


In [ ]:
# Plot MSE vs alpha
alphas = sorted(aug_results.keys())
val_mses = [aug_results[a]['best_mse'] for a in alphas]
baseline_mse = bc_result['best_mse']

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(alphas, val_mses, 'o-', linewidth=2, markersize=8, color='steelblue')
ax.axhline(y=baseline_mse, color='red', linestyle='--', linewidth=1.5, label=f'Pure BC (MSE={baseline_mse:.4f})')
ax.set_xlabel('α (synthetic mix ratio)')
ax.set_ylabel('Val MSE')
ax.set_title(f'C3: JEPA-Augmented BC (σ={best_sigma})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'c3_augmented_bc.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f'\n{"α":<8}{"Train MSE":<14}{"Val MSE":<14}{"Δ from baseline":<18}')
print('-' * 54)
for a in alphas:
    r = aug_results[a]
    delta = r['best_mse'] - baseline_mse
    print(f'{a:<8}{r["train_losses"][-1]:<14.6f}{r["best_mse"]:<14.6f}{delta:<+18.6f}')


## Section 6: C4 — REPA Alignment

### 6.1 Cache DINOv2 Intermediate Features

Run DINOv2 ViT-S/14 inference on all 4900 frames, extracting the CLS token at **all 12 transformer blocks**.
During REPA training, predictor layers $\{0,1,2,3\}$ are mapped to DINOv2 blocks $\{2,5,8,11\}$ (configurable via `cfg.repa_layer_map`).

This is the most computationally expensive step (~2–3h).


In [ ]:
import timm

print('Loading DINOv2 ViT-S/14 from timm...')
dino_model = timm.create_model('vit_small_patch14_reg4_dinov2.lvd142m', pretrained=True)
dino_model.eval()
dino_model = dino_model.to(device)

# Get transform for DINOv2 (224x224, ImageNet normalization)
dino_transform = timm.data.resolve_model_data_config(dino_model)
dino_transform = timm.data.create_transform(**dino_transform, is_training=False)

print(f'DINOv2 model loaded. Blocks: {len(dino_model.blocks)}')


In [ ]:
@torch.no_grad()
def extract_intermediate_features(dino_model, images_batch):
    """
    Extract CLS token at each transformer block for a batch of images.
    Returns: (B, 12, 384) tensor.
    """
    images_batch = images_batch.to(device)
    B = images_batch.shape[0]
    
    # Patch embedding
    x = dino_model.patch_embed(images_batch)
    x = dino_model._pos_embed(x)
    x = dino_model.norm_pre(x)
    
    features = torch.zeros(B, 12, cfg.embed_dim, device=device)
    for i, block in enumerate(dino_model.blocks):
        x = block(x)
        features[:, i] = x[:, 0]  # CLS token
    
    return features.cpu()


def cache_intermediate_features(image_dir, output_path, frame_count):
    """
    Cache DINOv2 intermediate features for all frames.
    Since we don't have raw images cached, we check if images are available.
    If not, this cell prints instructions for pre-computing via Colab.
    """
    # Check for saved images
    frames_path = EMBED_DIR / 'frames'
    if frames_path.exists():
        frames = sorted(frames_path.glob('*.pt'))
        if len(frames) >= frame_count:
            all_features = []
            bs = 32
            for i in tqdm(range(0, frame_count, bs), desc='Extracting DINOv2 intermediates'):
                batch = []
                for j in range(i, min(i+bs, frame_count)):
                    img = torch.load(frames[j], weights_only=False)
                    if img.dim() == 3:
                        img = img.unsqueeze(0)
                    batch.append(img)
                batch = torch.cat(batch, dim=0)
                feats = extract_intermediate_features(dino_model, batch)
                all_features.append(feats)
            all_features = torch.cat(all_features, dim=0)
            torch.save(all_features, output_path)
            print(f'  Saved {output_path} ({all_features.shape})')
            return all_features
    
    print(f'No pre-saved frames found. Intermediate feature caching requires raw images.')
    return None

# Try to cache intermediate features
dino_inter_train = cache_intermediate_features(
    EMBED_DIR / 'frames_train',
    EMBED_DIR / 'dino_intermediate_train.pt',
    train_emb.shape[0]
)

dino_inter_val = cache_intermediate_features(
    EMBED_DIR / 'frames_val',
    EMBED_DIR / 'dino_intermediate_val.pt',
    val_emb.shape[0]
)

if dino_inter_train is None:
    print('\n=== TO COMPUTE ON COLAB ===')
    print('Load images from Parquet, extract frames, and run:')
    print('  extract_intermediate_features(dino_model, image_batch)')
    print('  torch.save(features, EMBED_DIR / "dino_intermediate_train.pt")')


### 6.2 REPA Predictor Architecture

Extends `TransformerPredictor` to:
1. Return intermediate hidden states $h^{(0)}, h^{(1)}, h^{(2)}, h^{(3)}$ from each encoder layer
2. Attach projection heads $\pi_l = \mathrm{Linear}(d_\text{model}, 384)$ for each layer
3. Compute REPA loss: $\mathcal{L}_\text{REPA} = \sum_l (1 - \cos(\pi_l(h^{(l)}), f^{(l)}))$

Training loss: $\mathcal{L} = \mathcal{L}_\text{main}(\hat{z}, z^*) + \lambda \cdot \mathcal{L}_\text{REPA}$


In [ ]:
class REPAPredictor(nn.Module):
    """Transformer predictor with REPA-style intermediate alignment.
    
    Returns intermediate hidden states from each encoder layer for alignment
    with cached DINOv2 features.
    """
    def __init__(self, embed_dim=384, action_dim=7, d_model=256,
                 n_layers=4, n_heads=4, max_seq_len=64, dropout=0.2,
                 dino_layer_map=None):
        super().__init__()
        self.d_model = d_model
        self.n_layers = n_layers
        self.dino_layer_map = dino_layer_map or [2, 5, 8, 11]
        assert len(self.dino_layer_map) == n_layers
        
        self.embed_proj = nn.Linear(embed_dim, d_model)
        self.action_mlp = ActionMLP(action_dim, hidden_dim=128, out_dim=d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, max_seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, batch_first=True,
            dropout=dropout, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.predictor = nn.Linear(d_model, embed_dim)
        
        # REPA projection heads: one per predictor layer
        self.proj_heads = nn.ModuleList([
            nn.Linear(d_model, embed_dim) for _ in range(n_layers)
        ])
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, embeds, actions, return_intermediate=False):
        if embeds.dim() == 2:
            embeds = embeds.unsqueeze(1); actions = actions.unsqueeze(1)
        B, T, _ = embeds.shape
        x = self.embed_proj(embeds) + self.action_mlp(actions)
        x = x + self.pos_embed[:, :T, :]
        causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        
        # Manually step through encoder layers to capture intermediates
        intermediates = []
        for layer in self.transformer.layers:
            x = layer(x, src_mask=causal_mask)
            intermediates.append(x[:, -1, :])  # last token at each layer
        
        pred = self.predictor(x[:, -1, :])
        
        if return_intermediate:
            return pred, intermediates
        return pred

    def compute_repa_loss(self, intermediates, dino_features):
        """
        Compute REPA alignment loss.
        intermediates: list of (B, d_model) from each predictor layer
        dino_features: (B, 12, 384) or (B, n_dino_layers, 384)
        """
        total_loss = 0.0
        for l, h in enumerate(intermediates):
            dino_l = self.dino_layer_map[l]
            if dino_l >= dino_features.shape[1]:
                continue
            f_dino = dino_features[:, dino_l, :].to(h.device)
            proj = self.proj_heads[l](h)
            total_loss += (1.0 - F.cosine_similarity(proj, f_dino, dim=-1)).mean()
        return total_loss / len(intermediates)


class REPADataset(Dataset):
    """Extended ContextTripletDataset that also returns DINOv2 intermediate features."""
    def __init__(self, embed_path, action_path, triplet_path, dino_inter_path, T=8):
        self.embeddings = torch.load(embed_path)
        self.actions = torch.load(action_path)
        self.triplets = torch.load(triplet_path)
        self.dino_inter = torch.load(dino_inter_path)
        self.T = T
        valid = self.triplets[:, 0] >= (T - 1)
        self.triplets = self.triplets[valid]
        print(f'REPA dataset: {len(self.triplets)} samples (T={T})')

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        t_idx, a_idx, tpK_idx = self.triplets[idx]
        z_seq = self.embeddings[t_idx - self.T + 1 : t_idx + 1]
        a_seq = self.actions[t_idx - self.T + 1 : t_idx + 1]
        target = self.embeddings[tpK_idx]
        dino_feat = self.dino_inter[t_idx]  # (12, 384)
        if z_seq.shape[0] < self.T:
            pad = self.T - z_seq.shape[0]
            z_seq = torch.cat([torch.zeros(pad, self.embeddings.shape[1]), z_seq], dim=0)
            a_seq = torch.cat([torch.zeros(pad, self.actions.shape[1]), a_seq], dim=0)
        return z_seq, a_seq, target, dino_feat

print('REPA models and dataset ready.')


In [ ]:
@torch.no_grad()
def repa_compute_val_cosine(model, loader):
    model.eval()
    cs_list = []
    for z_seq, a_seq, target, _ in loader:
        z_seq = z_seq.to(device); a_seq = a_seq.to(device)
        target = target.to(device)
        if cfg.use_amp and device.type == 'cuda':
            with autocast():
                pred = model(z_seq, a_seq, return_intermediate=False)
        else:
            pred = model(z_seq, a_seq, return_intermediate=False)
        cs = F.cosine_similarity(pred, target, dim=-1).mean().item()
        cs_list.append(cs)
    return np.mean(cs_list)


def repa_train_one_epoch(model, loader, optimizer, scaler, global_step, total_steps, lambda_val):
    model.train()
    epoch_loss, epoch_cos, epoch_repa = 0.0, 0.0, 0.0
    for z_seq, a_seq, target, dino_feat in loader:
        z_seq = z_seq.to(device); a_seq = a_seq.to(device)
        target = target.to(device); dino_feat = dino_feat.to(device)
        lr = get_lr(global_step, cfg.warmup_steps, total_steps, cfg.lr)
        for pg in optimizer.param_groups:
            pg['lr'] = lr
        if cfg.use_amp and device.type == 'cuda':
            with autocast():
                pred, intermediates = model(z_seq, a_seq, return_intermediate=True)
                cos_loss = cosine_loss(pred, target)
                repa_loss = model.compute_repa_loss(intermediates, dino_feat)
                loss = cos_loss + lambda_val * repa_loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            pred, intermediates = model(z_seq, a_seq, return_intermediate=True)
            cos_loss = cosine_loss(pred, target)
            repa_loss = model.compute_repa_loss(intermediates, dino_feat)
            loss = cos_loss + lambda_val * repa_loss
            loss.backward()
            optimizer.step()
        optimizer.zero_grad()
        epoch_loss += loss.item()
        epoch_cos += cos_loss.item()
        epoch_repa += repa_loss.item()
        global_step += 1
    n = len(loader)
    return epoch_loss / n, epoch_cos / n, epoch_repa / n, global_step


def repa_train_model(model, train_ldr, val_ldr, lambda_val, epochs=None, verbose=True):
    epochs = epochs or cfg.repa_epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = GradScaler(enabled=(cfg.use_amp and device.type=='cuda'))
    total_steps = epochs * len(train_ldr)
    gs = 0
    train_losses, val_cos_list = [], []
    best_cos = -1.0
    best_state = None
    for epoch in range(1, epochs + 1):
        avg_loss, avg_cos, avg_repa, gs = repa_train_one_epoch(
            model, train_ldr, optimizer, scaler, gs, total_steps, lambda_val)
        val_cos = repa_compute_val_cosine(model, val_ldr)
        train_losses.append(avg_loss)
        val_cos_list.append(val_cos)
        if val_cos > best_cos:
            best_cos = val_cos
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if verbose and (epoch % max(1, epochs // 10) == 0 or epoch == 1 or epoch == epochs):
            print(f'E {epoch:3d}/{epochs} | loss={avg_loss:.4f} cos={1-avg_cos:.4f} repa={avg_repa:.4f} | val_cos={val_cos:.4f} | best={best_cos:.4f}')
    if best_state:
        model.load_state_dict(best_state)
    return {
        'model': model, 'val_cos': val_cos_list[-1], 'best_cos': best_cos,
        'train_losses': train_losses, 'val_cos_list': val_cos_list,
        'lambda': lambda_val
    }

print('REPA training utilities ready.')


In [ ]:
# REPA lambda sweep
repa_results = {}

# Check if intermediate features exist
dino_train_path = EMBED_DIR / 'dino_intermediate_train.pt'
dino_val_path = EMBED_DIR / 'dino_intermediate_val.pt'

if dino_train_path.exists() and dino_val_path.exists():
    print('DINOv2 intermediate features found. Starting REPA training...')
    print('=' * 60)
    
    for lam in cfg.repa_lambda_values:
        print(f'\nTraining REPA predictor with λ={lam}...')
        print('-' * 50)
        
        # Build loaders
        train_ds = REPADataset(
            EMBED_DIR / 'embeddings_train.pt',
            EMBED_DIR / 'actions_train.pt',
            EMBED_DIR / f'triplets_train_K1.pt',
            dino_train_path, T=cfg.best_T
        )
        val_ds = REPADataset(
            EMBED_DIR / 'embeddings_val.pt',
            EMBED_DIR / 'actions_val.pt',
            EMBED_DIR / f'triplets_val_K1.pt',
            dino_val_path, T=cfg.best_T
        )
        train_ldr = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                               num_workers=0, pin_memory=(device.type=='cuda'), drop_last=True)
        val_ldr = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                             num_workers=0, pin_memory=(device.type=='cuda'))
        
        model = REPAPredictor(
            embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
            d_model=cfg.d_model, n_layers=cfg.n_layers_tr,
            n_heads=cfg.n_heads, dropout=cfg.dropout_p,
            dino_layer_map=cfg.repa_layer_map,
        ).to(device)
        
        result = repa_train_model(model, train_ldr, val_ldr, lambda_val=lam,
                                  epochs=cfg.repa_epochs)
        repa_results[lam] = result
        
        # Save checkpoint
        ckpt = {
            'model_state_dict': model.state_dict(),
            'val_cos': result['val_cos'], 'best_cos': result['best_cos'],
            'lambda': lam, 'layer_map': cfg.repa_layer_map,
        }
        torch.save(ckpt, OUTPUT_DIR / f'module_c_repa_lambda{lam}.pt')
        print(f'  Saved outputs/module_c_repa_lambda{lam}.pt (best_cos={result["best_cos"]:.4f})')

else:
    print('DINOv2 intermediate features not found.')
    print('Run Section 6.1 on Colab first to cache:')
    print(f'  Expected: {dino_train_path}')
    print(f'  Expected: {dino_val_path}')
    print('Skipping REPA training for now.')


In [ ]:
if repa_results:
    fig, ax = plt.subplots(figsize=(9, 5))
    
    # Baseline: best Transformer K=1 T=8 val_cos from module_a
    baseline_cos = ckpt.get('val_cos', 0.9518)
    ax.axhline(y=baseline_cos, color='gray', linestyle='--', linewidth=1.5,
               label=f'Baseline (no REPA): {baseline_cos:.4f}')
    
    colors = ['#e74c3c', '#2ecc71', '#3498db']
    for i, lam in enumerate(sorted(repa_results.keys())):
        r = repa_results[lam]
        ax.plot(r['val_cos_list'], color=colors[i], linewidth=2,
                label=f'λ={lam} (best={r["best_cos"]:.4f})')
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val Cosine Similarity')
    ax.set_title('C4: REPA Training Curves (Predictor with Intermediate Alignment)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'c4_repa_training.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Summary
    print(f'\n{"λ":<8}{"Best CosSim":<14}{"Final CosSim":<14}{"Δ baseline":<14}')
    print('-' * 50)
    for lam in sorted(repa_results.keys()):
        r = repa_results[lam]
        delta = r['best_cos'] - baseline_cos
        print(f'{lam:<8}{r["best_cos"]:<14.4f}{r["val_cos"]:<14.4f}{delta:<+14.4f}')
else:
    print('No REPA results to plot (intermediate features not cached).')


## Section 7: C5 — REPA Ablation Results

Compare baseline predictor (no REPA) vs best-λ REPA predictor on three axes:
1. **Cosine similarity** on val triplets at $K \in \{1,5,10,20\}$
2. **Rollout drift**: autoregressive rollout cos_sim vs step
3. **CEM success**: flat CEM planning performance


In [ ]:
if repa_results:
    # Find best lambda
    best_lam = max(repa_results, key=lambda k: repa_results[k]['best_cos'])
    print(f'Best REPA lambda: {best_lam} (cos={repa_results[best_lam]["best_cos"]:.4f})')
    
    # Load baseline predictor (no REPA, K=1, T=8)
    baseline_model = TransformerPredictor(
        embed_dim=cfg.embed_dim, action_dim=cfg.action_dim, d_model=cfg.d_model,
        n_layers=cfg.n_layers_tr, n_heads=cfg.n_heads, dropout=cfg.dropout_p,
    ).to(device)
    baseline_model.load_state_dict(ckpt['model_state_dict'])
    baseline_model.eval()
    
    # Load REPA predictor (best lambda)
    repa_model = REPAPredictor(
        embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
        d_model=cfg.d_model, n_layers=cfg.n_layers_tr,
        n_heads=cfg.n_heads, dropout=cfg.dropout_p,
        dino_layer_map=cfg.repa_layer_map,
    ).to(device)
    repa_ckpt = torch.load(OUTPUT_DIR / f'module_c_repa_lambda{best_lam}.pt',
                          map_location=device, weights_only=False)
    repa_model.load_state_dict(repa_ckpt['model_state_dict'])
    repa_model.eval()
    
    print('Both predictors loaded for comparison.')
else:
    print('Skipping C5: REPA results not available.')
    print('Loading baseline for reference metrics only.')
    baseline_model = TransformerPredictor(
        embed_dim=cfg.embed_dim, action_dim=cfg.action_dim, d_model=cfg.d_model,
        n_layers=cfg.n_layers_tr, n_heads=cfg.n_heads, dropout=cfg.dropout_p,
    ).to(device)
    baseline_model.load_state_dict(ckpt['model_state_dict'])
    baseline_model.eval()


In [ ]:
# Evaluation utility: compute val cosine similarity for a given K
@torch.no_grad()
def eval_val_cos(model, K, T=8):
    val_ds = ContextTripletDataset(
        EMBED_DIR / 'embeddings_val.pt',
        EMBED_DIR / 'actions_val.pt',
        EMBED_DIR / f'triplets_val_K{K}.pt', T=T
    )
    val_ldr = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                         num_workers=0, pin_memory=(device.type=='cuda'))
    cs_list = []
    for z_seq, a_seq, target in val_ldr:
        z_seq = z_seq.to(device); a_seq = a_seq.to(device); target = target.to(device)
        pred = model(z_seq, a_seq, return_intermediate=False)
        cs = F.cosine_similarity(pred, target, dim=-1).mean().item()
        cs_list.append(cs)
    return np.mean(cs_list)


# Need ContextTripletDataset for evaluation
class ContextTripletDataset(Dataset):
    def __init__(self, embed_path, action_path, triplet_path, T=1):
        self.embeddings = torch.load(embed_path)
        self.actions = torch.load(action_path)
        self.triplets = torch.load(triplet_path)
        self.T = T
        valid = self.triplets[:, 0] >= (T - 1)
        self.triplets = self.triplets[valid]
    def __len__(self): return len(self.triplets)
    def __getitem__(self, idx):
        t_idx, a_idx, tpK_idx = self.triplets[idx]
        z_seq = self.embeddings[t_idx - self.T + 1 : t_idx + 1]
        a_seq = self.actions[t_idx - self.T + 1 : t_idx + 1]
        target = self.embeddings[tpK_idx]
        if z_seq.shape[0] < self.T:
            pad = self.T - z_seq.shape[0]
            z_seq = torch.cat([torch.zeros(pad, self.embeddings.shape[1]), z_seq], dim=0)
            a_seq = torch.cat([torch.zeros(pad, self.actions.shape[1]), a_seq], dim=0)
        return z_seq, a_seq, target


if repa_results:
    print('Cosine similarity comparison (baseline vs REPA)...')
    results_cos = {'K': [], 'baseline': [], 'repa': []}
    for K in cfg.K_values:
        cs_base = eval_val_cos(baseline_model, K, cfg.best_T)
        cs_repa = eval_val_cos(repa_model, K, cfg.best_T)
        results_cos['K'].append(K)
        results_cos['baseline'].append(cs_base)
        results_cos['repa'].append(cs_repa)
        print(f'  K={K:2d}: baseline={cs_base:.4f}  REPA={cs_repa:.4f}  Δ={cs_repa-cs_base:+.4f}')

    # Bar chart
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(cfg.K_values))
    w = 0.35
    ax.bar(x - w/2, results_cos['baseline'], w, label='Baseline (no REPA)', color='gray')
    ax.bar(x + w/2, results_cos['repa'], w, label=f'REPA (λ={best_lam})', color='steelblue')
    ax.set_xticks(x); ax.set_xticklabels([f'K={k}' for k in cfg.K_values])
    ax.set_ylabel('Val Cosine Similarity')
    ax.set_title('C5: Cosine Similarity — Baseline vs REPA')
    ax.legend(); ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'c5_cos_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Skipping: REPA predictor not available.')


In [ ]:
@torch.no_grad()
def autoregressive_rollout(model, z_start, actions, K, T, max_steps=20, is_repa=False):
    """Rollout supporting both regular and REPA predictors."""
    model.eval()
    z_context = z_start.unsqueeze(0).unsqueeze(0).repeat(1, T, 1).to(device)
    a_context = actions[:1].unsqueeze(0).repeat(1, T, 1).to(device)
    pred_embs, true_embs, cos_sims = [z_start.cpu().numpy()], [z_start.cpu().numpy()], [1.0]
    current_step = 0
    for m in range(1, max_steps + 1):
        target_step = current_step + K
        if target_step >= len(actions):
            break
        if is_repa:
            pred = model(z_context.to(device), a_context.to(device), return_intermediate=False)
        else:
            pred = model(z_context.to(device), a_context.to(device))
        z_context = torch.cat([z_context[:, 1:, :], pred.unsqueeze(1)], dim=1)
        a_new = actions[target_step:target_step+1].unsqueeze(0).repeat(1, T, 1).to(device)
        a_context = a_new
        pred_embs.append(pred.squeeze(0).cpu().numpy())
        true_embs.append(val_emb[target_step].cpu().numpy())
        cs = F.cosine_similarity(pred.squeeze(0), val_emb[target_step].to(device), dim=0).item()
        cos_sims.append(cs)
        current_step = target_step
    return pred_embs, true_embs, cos_sims


# Run rollout comparison
if repa_results:
    print('Rollout drift comparison...')
    start_idx = 0
    z_0 = val_emb[start_idx]
    actions_seq = val_act[start_idx:start_idx + 4 * 20 + cfg.best_T]
    
    fig, ax = plt.subplots(figsize=(8, 5))
    for K in [1, 5]:
        _, _, cos_base = autoregressive_rollout(baseline_model, z_0, actions_seq,
                                                K=K, T=cfg.best_T, max_steps=10, is_repa=False)
        _, _, cos_repa = autoregressive_rollout(repa_model, z_0, actions_seq,
                                                K=K, T=cfg.best_T, max_steps=10, is_repa=True)
        ax.plot(cos_base, 'o-', markersize=4, linewidth=1.5,
                label=f'Baseline K={K}')
        ax.plot(cos_repa, 's--', markersize=4, linewidth=1.5,
                label=f'REPA K={K}')
    
    ax.set_xlabel('Rollout step')
    ax.set_ylabel('Cosine Similarity')
    ax.set_title('C5: Rollout Drift — Baseline vs REPA')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'c5_rollout_drift.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Skipping: REPA predictor not available.')


In [ ]:
# Flat CEM planner (adapted from Module B)
@torch.no_grad()
def flat_cem_plan(z_start, z_target, predictor, horizon, K, T,
                  n_pop=128, elite_frac=0.1, n_iters=5, noise_scale=0.5, is_repa=False):
    n_elite = max(1, int(n_pop * elite_frac))
    mu = torch.zeros(horizon, cfg.action_dim, device=device)
    sigma = torch.ones(horizon, cfg.action_dim, device=device) * noise_scale
    best_seq, best_score = None, -float('inf')
    
    for it in range(n_iters):
        eps = torch.randn(n_pop, horizon, cfg.action_dim, device=device)
        actions = mu.unsqueeze(0) + sigma.unsqueeze(0) * eps
        scores = torch.zeros(n_pop, device=device)
        
        for i in range(n_pop):
            z_ctx = z_start.unsqueeze(0).unsqueeze(0).repeat(1, T, 1).to(device)
            a_ctx = torch.zeros(1, T, cfg.action_dim, device=device)
            a_ctx[:, -1, :] = actions[i, 0]
            for step in range(horizon):
                a_cur = actions[i, step:step+1].unsqueeze(0)
                a_ctx = torch.cat([a_ctx[:, 1:, :], a_cur], dim=1)
                if is_repa:
                    pred = predictor(z_ctx, a_ctx, return_intermediate=False)
                else:
                    pred = predictor(z_ctx, a_ctx)
                z_ctx = torch.cat([z_ctx[:, 1:, :], pred.unsqueeze(1)], dim=1)
            z_pred = z_ctx[:, -1, :].squeeze(0)
            scores[i] = F.cosine_similarity(z_pred.unsqueeze(0), z_target.unsqueeze(0), dim=-1)
        
        elite_idx = torch.topk(scores, n_elite).indices
        ea = actions[elite_idx]
        mu = ea.mean(dim=0)
        sigma = ea.std(dim=0) + 1e-6
        iter_best = scores[elite_idx[0]].item()
        if iter_best > best_score:
            best_score = iter_best; best_seq = actions[elite_idx[0]].cpu().clone()
    return best_seq, best_score


if repa_results:
    print('CEM success comparison...')
    # Test on a few val episodes
    test_ep_indices = list(range(cfg.n_train_episodes, min(cfg.n_train_episodes + 10, len(parquet_files))))
    
    cem_results = {'episode': [], 'baseline_cos': [], 'repa_cos': []}
    for ep_idx in tqdm(test_ep_indices, desc='CEM eval'):
        split_name, start = ep_to_emb_start[ep_idx]
        emb = val_emb if split_name == 'val' else train_emb
        n_frames = all_ep_n_frames[ep_idx]
        if n_frames < 20:
            continue
        z_start = emb[start].to(device)
        z_target = emb[start + n_frames - 1].to(device)
        
        _, score_base = flat_cem_plan(z_start, z_target, baseline_model,
                                      horizon=20, K=1, T=cfg.best_T, is_repa=False)
        _, score_repa = flat_cem_plan(z_start, z_target, repa_model,
                                      horizon=20, K=1, T=cfg.best_T, is_repa=True)
        cem_results['episode'].append(ep_idx)
        cem_results['baseline_cos'].append(score_base)
        cem_results['repa_cos'].append(score_repa)
    
    avg_base = np.mean(cem_results['baseline_cos'])
    avg_repa = np.mean(cem_results['repa_cos'])
    print(f'\nAvg CEM goal cos: baseline={avg_base:.4f}  REPA={avg_repa:.4f}')
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(['Baseline', 'REPA'], [avg_base, avg_repa], color=['gray', 'steelblue'])
    ax.set_ylabel('Avg CEM Goal Cosine Similarity')
    ax.set_title('C5: CEM Planning Success')
    for i, v in enumerate([avg_base, avg_repa]):
        ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')
    ax.set_ylim(0, 1); ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'c5_cem_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Skipping: REPA predictor not available.')


In [ ]:
print('=' * 60)
print('Module C Complete')
print('=' * 60)

print(f'\nCheckpoints saved:')
import glob
for f in sorted(OUTPUT_DIR.glob('module_c_*')):
    print(f'  {f.name}')

print(f'\nFigures saved:')
for f in sorted(OUTPUT_DIR.glob('c[1-5]_*.png')):
    print(f'  {f.name}')
